# 05 - Model evaluation, HMM/Viterbi and maps
Model comparison plots, confusion matrices, decoder tuning (training data via out-of-fold probabilities),
classifier-only vs HMM on the **student-testing** trips, and Folium maps.

In [ ]:
import sys, pathlib, warnings
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt

OUT, GRAPHS, MAPS = ROOT / "outputs", ROOT / "outputs" / "graphs", ROOT / "outputs" / "maps"
PROC = ROOT / "dataset" / "processed"
for d in (GRAPHS, MAPS, PROC, ROOT / "models" / "configurations"):
    d.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 200, "display.max_columns", 40)

In [ ]:
import json, joblib
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from src import data_loader as dl
from src.evaluation import *
from src.feature_engineering import FEATURE_COLUMNS, load_dataset
from src.models import get_model
from src.road_network import RoadNetwork
from src.route_reconstruction.hmm import HMMConfig, RouteHMM, tune_hmm

tr_c, tr_o, CFG = load_dataset(PROC / "training_features.joblib")
st_c, st_o, _ = load_dataset(PROC / "student_testing_features.joblib")
scaler = joblib.load(ROOT / "models" / "scaler.pkl")
final_model = joblib.load(ROOT / "models" / "final_model.pkl")
pcfg = json.load(open(ROOT / "models" / "configurations" / "pipeline_config.json"))
Xtr = scaler.transform(tr_c[FEATURE_COLUMNS]).astype(np.float32); Xst = scaler.transform(st_c[FEATURE_COLUMNS]).astype(np.float32)
comp = pd.read_csv(OUT / "model_comparison_full.csv"); cvf = pd.read_csv(OUT / "cv_fold_results.csv")
print("final model:", pcfg["model_name"], pcfg["model_params"])

## 1. Model comparison plots

In [ ]:
def bars(ax, vals, labels, title, ylabel, err=None, color="#4c78a8"):
    ax.bar(labels, vals, yerr=err, color=color, capsize=3); ax.set(title=title, ylabel=ylabel); ax.tick_params(axis="x", rotation=30)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
bars(ax[0], comp.cv_macro_f1, comp.model, "Cross-validation macro F1 (mean +- std over folds, training trips)", "macro F1", err=comp.cv_macro_f1_std)
ax[0].set_ylim(comp.cv_macro_f1.min() - 0.03, min(1, comp.cv_macro_f1.max() + 0.02))
w = 0.38; x = np.arange(len(comp))
ax[1].bar(x - w / 2, comp.student_top1_acc, w, label="Top-1 accuracy"); ax[1].bar(x + w / 2, comp.student_macro_f1, w, label="Macro F1")
ax[1].set_xticks(x); ax[1].set_xticklabels(comp.model, rotation=30); ax[1].set_title("Student-testing"); ax[1].legend()
ax[1].set_ylim(min(comp.student_top1_acc.min(), comp.student_macro_f1.min()) - 0.05, 1)
bars(ax[2], comp.student_log_loss, comp.model, "Student-testing log loss (lower is better)", "log loss", color="#e45756")
fig.tight_layout(); fig.savefig(GRAPHS / "model_comparison.png"); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
w = 0.27
for k, (col, lab) in enumerate([("train_top1_acc", "train"), ("cv_top1_acc", "CV"), ("student_top1_acc", "student-test")]):
    ax[0].bar(x + (k - 1) * w, comp[col], w, label=lab)
ax[0].set_xticks(x); ax[0].set_xticklabels(comp.model, rotation=30); ax[0].set_title("Top-1 accuracy: over/under-fitting check"); ax[0].legend()
ax[0].set_ylim(comp[["train_top1_acc", "cv_top1_acc", "student_top1_acc"]].min().min() - 0.05, 1)
ax[1].scatter(comp.inference_ms_per_1k_obs, comp.student_top1_acc)
for _, r in comp.iterrows(): ax[1].annotate(r.model, (r.inference_ms_per_1k_obs, r.student_top1_acc), fontsize=8)
ax[1].set(xscale="log", xlabel="inference ms per 1000 observations", ylabel="student top-1 accuracy", title="Accuracy vs inference cost")
fig.tight_layout(); fig.savefig(GRAPHS / "model_fit_and_speed.png"); plt.show()

In [ ]:
# per-fold stability
cvf.pivot(index="fold", columns="model", values="macro_f1").round(4)

## 2. Confusion matrices (student-testing, candidate level, top-1 decision)

In [ ]:
names = list(comp.model)
fig, ax = plt.subplots(1, len(names), figsize=(3.6 * len(names), 3.6))
ax = np.atleast_1d(ax)
lab = st_c.has_ref.to_numpy(); y = st_c.y.to_numpy()[lab]
for a, n in zip(ax, names):
    p = np.load(PROC / f"student_proba_{n}.npy"); dec = np.zeros(len(st_c), dtype=int); dec[top1_decision(st_c, p)] = 1
    ConfusionMatrixDisplay(confusion_matrix(y, dec[lab])).plot(ax=a, colorbar=False, values_format="d"); a.set_title(n, fontsize=9); a.grid(False)
fig.tight_layout(); fig.savefig(GRAPHS / "confusion_matrices.png"); plt.show()

In [ ]:
imp = getattr(final_model, "feature_importances_", None)
if imp is None and hasattr(final_model, "coef_"): imp = np.abs(final_model.coef_[0])
if imp is not None:
    s = pd.Series(imp, index=FEATURE_COLUMNS).sort_values()
    fig, ax = plt.subplots(figsize=(6, 6)); s.plot.barh(ax=ax); ax.set_title(f"Feature importance: {pcfg['model_name']}"); fig.tight_layout()
    fig.savefig(GRAPHS / "feature_importance.png"); plt.show()

## 3. HMM / Viterbi decoder
Hidden states = candidate arcs of each observation; emissions = classifier probabilities; transitions = network
connectivity (see `src/route_reconstruction/hmm.py`). Decoder parameters are tuned with **out-of-fold**
probabilities on training trips (the classifier never saw those trips), so the student-testing trips stay a clean check.

In [ ]:
mod = get_model(pcfg["model_name"])
oof = oof_proba(lambda: mod.build(**pcfg["model_params"]), Xtr, tr_c, n_splits=3, seed=42,
                max_train_rows=mod.MAX_TRAIN_ROWS or 400_000)
rng = np.random.default_rng(7)
tune_groups = rng.choice(tr_c.group.unique(), size=min(15, tr_c.group.nunique()), replace=False)
def trip_input(c_all, o_all, proba, g, partition):
    c = c_all[c_all.group == g]; o = o_all[o_all.group == g].sort_values("obs_idx")
    trip = dl.load_trip(partition, f"{int(g):08d}", with_route=False)
    return {"network": RoadNetwork(trip.nodes, trip.arcs), "cands": c, "proba": proba[(c_all.group == g).to_numpy()], "obs": o}
tune_inputs = [trip_input(tr_c, tr_o, oof, g, "training") for g in tune_groups]
best_hmm, hmm_trials = tune_hmm(tune_inputs, n_iter=30, seed=42)
hmm_trials.to_csv(OUT / "hmm_tuning_trials.csv", index=False)
print(best_hmm); hmm_trials.sort_values("top1_acc", ascending=False).head(5).round(4)

## 4. Classifier-only vs classifier + HMM (student-testing)

In [ ]:
p_st = final_model.predict_proba(Xst)[:, 1]
rows = []
for g in sorted(st_c.group.unique()):
    ti = trip_input(st_c, st_o, p_st, g, "student_testing")
    o, c, net = ti["obs"], ti["cands"], ti["network"]
    ref = o.ref_arc.to_numpy()
    ref_route = dl.load_reference_route("student_testing", f"{int(g):08d}")
    clf = predicted_arc_per_obs(c.assign(obs_key=c.obs_idx), ti["proba"], o.assign(obs_key=o.obs_idx))
    variants = {"classifier_only": clf,
                "hmm_default": RouteHMM(net, HMMConfig()).decode(c, ti["proba"], o.t.to_numpy()),
                "hmm_tuned": RouteHMM(net, best_hmm).decode(c, ti["proba"], o.t.to_numpy())}
    for name, arcs in variants.items():
        rows.append({"trip": g, "method": name, "n_obs": len(o), "top1_acc": float(np.mean((arcs == ref) & (ref >= 0))),
                     **route_continuity(arcs, net), **route_overlap(arcs, ref_route)})
hmm_cmp = pd.DataFrame(rows); hmm_cmp.to_csv(OUT / "hmm_comparison.csv", index=False)
agg = hmm_cmp.groupby("method").apply(lambda d: pd.Series({
    "top1_acc (obs-weighted)": np.average(d.top1_acc, weights=d.n_obs),
    "route_continuity (pooled)": 1 - d.n_breaks.sum() / max(d.n_transitions.sum(), 1),
    "route_f1 (mean over trips)": d.route_f1.mean(), "n_breaks": d.n_breaks.sum()}))
agg.round(4)

In [ ]:
piv = hmm_cmp.pivot(index="trip", columns="method", values="top1_acc")
piv["delta_tuned_vs_clf"] = piv.hmm_tuned - piv.classifier_only
print(piv.round(4)); print("trips where HMM did NOT improve top-1:", piv.index[piv.delta_tuned_vs_clf <= 0].tolist())
fig, ax = plt.subplots(figsize=(9, 3.8)); piv[["classifier_only", "hmm_default", "hmm_tuned"]].plot.bar(ax=ax)
ax.set(ylabel="top-1 accuracy", title="Student-testing: classifier-only vs HMM per trip"); ax.set_ylim(piv.min().min() - 0.05, 1)
fig.tight_layout(); fig.savefig(GRAPHS / "hmm_vs_classifier.png"); plt.show()

In [ ]:
pcfg["hmm_config"] = best_hmm.to_dict()
json.dump(pcfg, open(ROOT / "models" / "configurations" / "pipeline_config.json", "w"), indent=2)
print("tuned decoder parameters saved to pipeline_config.json")

## 5. End-to-end check through the deployment code path (`Predictor`)

In [ ]:
from src.pipeline import Predictor
P = Predictor()
tid = dl.list_trips("student_testing")[0]
res = P.predict_trip(dl.load_trip("student_testing", tid, with_route=False))   # no labels given to the predictor
sc = P.score(res, dl.load_reference_route("student_testing", tid))             # scoring is separate
print(tid, {k: round(v, 4) for k, v in sc.items() if k in ("classifier_top1_acc", "hmm_top1_acc", "classifier_route_continuity", "hmm_route_continuity")})
print(res["timing_s"])

## 6. Folium maps (partition: student-testing)

In [ ]:
# FOLIUM
from src.visualisation import build_trip_map
for tid in dl.list_trips("student_testing")[:3]:
    trip = dl.load_trip("student_testing", tid)
    res = P.predict_trip(trip); net = res["network"]
    lat, lon = res["obs"].lat.to_numpy(), res["obs"].lon.to_numpy()
    src_lat = net.nodes_lonlat[net.src][:, 1]; src_lon = net.nodes_lonlat[net.src][:, 0]
    near = np.flatnonzero((src_lat > lat.min() - .002) & (src_lat < lat.max() + .002) & (src_lon > lon.min() - .002) & (src_lon < lon.max() + .002))[:8000]
    m = build_trip_map(res["obs"][["lat", "lon"]].to_numpy().tolist(), network_segments=net.arc_segments_latlon(near),
                       reference_polylines=net.arcs_to_polylines(trip.route),
                       classifier_polylines=net.arcs_to_polylines(res["classifier_arcs"]),
                       hmm_polylines=net.arcs_to_polylines(res["hmm_arcs"]), title=f"Student-test trip {tid}")
    m.save(MAPS / f"student_{tid}_prediction.html"); print("saved", tid)

## 7. Discussion (fill in from your results)
* Which model won and why (primary metric, stability across folds, probability quality, inference time):
* Did tuning help? By how much, at what cost?
* Where did the HMM help and where did it not? 